In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

if api_key:
    print("✅ API Key loaded successfully!")
else:
    print("❌ API Key not found. Check your .env file.")

✅ API Key loaded successfully!


In [7]:
pip install langchain --upgrade

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [9]:
# Load the PDF
loader = PyPDFLoader("sample.pdf")
pages = loader.load()

print(f"✅ PDF loaded successfully!")
print(f"   Total pages : {len(pages)}")
print(f"\nPreview of page 1:")
print(pages[0].page_content[:500])

✅ PDF loaded successfully!
   Total pages : 2

Preview of page 1:
Module-1 
Q.1 Define Customer Relationship Management (CRM). Explain its meaning, definitions, 
importance, and growth. Discuss how CRM is used in modern organizations with suitable 
examples. 
Q.2 Explain the concept of E-CRM (Electronic Customer Relationship Management). 
Discuss the role of CRM and ECRM in digital business environments. Also explain the 
nature and characteristics of E-CRM. 
Q.3 Describe the CRM Framework and CRM Cycle in detail. Explain the stages involved in 
the CRM proces


In [10]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # each chunk has max 1000 characters
    chunk_overlap=200     # chunks overlap by 200 characters to avoid losing context
)

chunks = text_splitter.split_documents(pages)

print(f"✅ PDF split into chunks!")
print(f"   Total pages  : {len(pages)}")
print(f"   Total chunks : {len(chunks)}")
print(f"\nPreview of chunk 1:")
print(chunks[0].page_content[:300])

✅ PDF split into chunks!
   Total pages  : 2
   Total chunks : 4

Preview of chunk 1:
Module-1 
Q.1 Define Customer Relationship Management (CRM). Explain its meaning, definitions, 
importance, and growth. Discuss how CRM is used in modern organizations with suitable 
examples. 
Q.2 Explain the concept of E-CRM (Electronic Customer Relationship Management). 
Discuss the role of CRM a


Split PDF into Chunks

A PDF can be very long, and sending the entire document to Gemini at once would be 
too expensive and slow. Instead, we break it into smaller overlapping pieces called **chunks**.

**chunk_size=1000** → Each chunk contains a maximum of 1000 characters.

**chunk_overlap=200** → Consecutive chunks share 200 characters with each other. 
This is important because if an answer spans across two chunks, the overlap ensures 
we don't lose that context.

**Example:**
- Chunk 1 → characters 1 to 1000
- Chunk 2 → characters 800 to 1800  ← starts 200 characters before Chunk 1 ends
- Chunk 3 → characters 1600 to 2600

This way no information is lost at the boundaries between chunks.

**RecursiveCharacterTextSplitter** is smart about where it splits — it tries to 
split at natural boundaries like paragraphs and sentences rather than cutting 
words in the middle.

In [15]:
import google.generativeai as genai

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

for model in genai.list_models():
    if "embedContent" in model.supported_generation_methods:
        print(model.name)

C:\Users\dhruv\AppData\Local\Temp\ipykernel_20868\1476907710.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


In [16]:
# Create embeddings using Google's embedding model
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

# Store the chunks and their embeddings in FAISS vector database
vectorstore = FAISS.from_documents(chunks, embeddings)

print("✅ Embeddings created and stored in FAISS!")
print(f"   Total vectors stored : {vectorstore.index.ntotal}")

✅ Embeddings created and stored in FAISS!
   Total vectors stored : 4


Create Embeddings and Store in FAISS

Each chunk of text is converted into a **vector** (a list of numbers) using 
Google's embedding model. Similar pieces of text will have similar vectors.

These vectors are then stored in a **FAISS vector database**, which allows 
us to quickly search for the most relevant chunks when a question is asked.

Think of it as creating an index of your PDF — just like the index at the 
back of a textbook that helps you find topics quickly.

In [25]:
# Initialize Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0.3
)

# Create a retriever from the vector store
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}  # retrieve top 3 most relevant chunks
)

# Create the prompt template
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context provided below.
If you don't know the answer from the context, say "I don't have enough information to answer this."

Context: {context}

Question: {question}

Answer:
""")

# Build the RAG chain
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG chain is ready!")

✅ RAG chain is ready!


Set up Gemini and Build the RAG Chain

This cell connects all the pieces together:

- **Gemini LLM** → the AI model that generates the final answer
- **Retriever** → searches FAISS for the top 3 most relevant chunks for a given question
- **Prompt Template** → instructs Gemini to answer only from the retrieved context
- **RAG Chain** → the pipeline: Question → Retrieve chunks → Send to Gemini → Get answer

**temperature=0.3** means the model gives consistent, factual answers 
rather than creative ones (0 = fully deterministic, 1 = very creative).

In [26]:
question = "What is CRM?"
response = rag_chain.invoke(question)

print(f"Question: {question}")
print(f"\nAnswer: {response}")

Question: What is CRM?

Answer: I don't have enough information to answer this.


In [19]:
for model in genai.list_models():
    if "generateContent" in model.supported_generation_methods:
        print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-max-preview-04-2026
models/deep-research-prev

In [23]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
print("✅ New API key loaded!")

✅ New API key loaded!


In [27]:
# First let's see what's actually in our chunks
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ---")
    print(chunk.page_content[:200])
    print()

--- Chunk 1 ---
Module-1 
Q.1 Define Customer Relationship Management (CRM). Explain its meaning, definitions, 
importance, and growth. Discuss how CRM is used in modern organizations with suitable 
examples. 
Q.2 Ex

--- Chunk 2 ---
customer touch points effectively? Discuss the success factors in Online Supply Chain 
Management and e-Customer Relationship Management. 
 
Module-2 
Q.1 Explain the role of technology in Customer Re

--- Chunk 3 ---
consider while identifying the right CRM program? Discuss with suitable examples. 
Q.5 Compare and contrast the application of CRM in B2C and B2B markets. Highlight the 
key differences in strategies,

--- Chunk 4 ---
Q.3 Describe the architecture and key components of an E-CRM system. Explain how these 
components interact to enhance customer relationship management in online businesses. 
Q.4 Discuss the major iss



In [28]:
question = "What is the role of technology in Customer Relationship Management?"
response = rag_chain.invoke(question)

print(f"Question: {question}")
print(f"\nAnswer: {response}")

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key expired. Please renew the API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key expired. Please renew the API key.'}]}}

In [30]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0.3
)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ LLM reinitialized with new key!")

✅ LLM reinitialized with new key!


In [31]:
question = "What is the role of technology in Customer Relationship Management?"
response = rag_chain.invoke(question)
print(f"Question: {question}")
print(f"\nAnswer: {response}")

Question: What is the role of technology in Customer Relationship Management?

Answer: I don't have enough information to answer this.


In [38]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

# Reinitialize embeddings
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

# Reinitialize LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0.3
)

print("✅ Reinitialized with new key!")

✅ Reinitialized with new key!


In [39]:
# Load PDF
loader = PyPDFLoader("sample2.pdf")
pages = loader.load()

# Split into chunks
chunks = text_splitter.split_documents(pages)

# Create new vectorstore
vectorstore = FAISS.from_documents(chunks, embeddings)

# Rebuild retriever and chain
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(f"✅ Pipeline ready!")
print(f"   Total pages  : {len(pages)}")
print(f"   Total chunks : {len(chunks)}")

✅ Pipeline ready!
   Total pages  : 18
   Total chunks : 8


In [40]:
question = "What is this document about?"
response = rag_chain.invoke(question)

print(f"Question: {question}")
print(f"\nAnswer: {response}")

Question: What is this document about?

Answer: This document is about an experiment. The aim of the experiment is to create a Virtual Private Cloud (VPC) in AWS and then create an EC2 instance in the same VPC.


In [41]:
questions = [
    "What is a VPC?",
    "What are the steps to create an EC2 instance?",
    "What is the aim of this experiment?"
]

for q in questions:
    print(f"Question: {q}")
    response = rag_chain.invoke(q)
    print(f"Answer: {response}")
    print()

Question: What is a VPC?
Answer: I don't have enough information to answer this.

Question: What are the steps to create an EC2 instance?
Answer: The steps to create an EC2 instance are:

1.  Navigate to EC2 Dashboard.
2.  Click on Launch Instance.
3.  Choose an Amazon Machine Image (AMI).
4.  Select instance type (e.g., t2.micro).
5.  Under network settings, select the created VPC and subnet.
6.  Configure security group (allow SSH/HTTP).
7.  Create or select a key pair.
8.  Review instance configuration.
9.  Click on Launch Instance.
10. Verify EC2 instance is running in the same VPC.

Question: What is the aim of this experiment?
Answer: The aim of this experiment is to create a Virtual Private Cloud in AWS and create an EC2 instance in the same VPC.

